# 02 — Candlestick Image Inspection

**STOP POINT**: Render 5 sample images and visually confirm before training the CNN.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.utils.seeds import set_all_seeds
set_all_seeds()
%matplotlib inline

## 1. Load OHLCV for Sector ETFs

In [ ]:
import yaml
from src.data.alpaca_loader import load_universe
with open('../config/assets.yaml') as f:
    assets = yaml.safe_load(f)

tickers = assets['sector_etfs'][:5]  # Inspect first 5 sectors
ohlcv = load_universe(tickers, start='2018-01-01')
print({t: df.shape for t, df in ohlcv.items()})

## 2. Produce 5 Sample Images (STOP POINT)

Save to `results/figures/candle_samples/` -- inspect visually before continuing.

In [ ]:
from src.data.candlestick_render import produce_sample_images
sample_paths = produce_sample_images(
    tickers=list(ohlcv.keys()),
    ohlcv_dict=ohlcv,
    n_samples=5,
    seed=42,
)
print('Sample images saved:')
for p in sample_paths:
    print(' ', p)

## 3. Display Samples Inline

In [ ]:
from PIL import Image
fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 4))
if len(sample_paths) == 1:
    axes = [axes]
for ax, p in zip(axes, sample_paths):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
plt.suptitle('Candlestick Samples — STOP: Verify Visually Before CNN Training')
plt.tight_layout()
plt.show()

## 4. Render All Images (Cache) for CNN Training

In [ ]:
from src.data.candlestick_render import render_and_cache
# After visual inspection passes, render all images for all tickers
all_tickers = assets['sector_etfs'] + assets['indices'] + assets['commodities'] + assets['bond_etfs']
ohlcv_all = load_universe(all_tickers, start='2016-01-01')

for ticker, df in ohlcv_all.items():
    saved = render_and_cache(ticker, df, window=60, force_refresh=False)
    print(f'{ticker}: {len(saved)} images cached')